# These are the examples and exercises for Lesson 2: "Topic Modelling".

In [2]:
import string
from collections import Counter
from pprint import pprint
import gzip
import matplotlib.pyplot as plt 
import numpy as np
from numpy.linalg import svd
from numpy import diag

%matplotlib inline

Fetch stop words from a standard set; save them in a set object to remove duplicates from the source and to make lookups quick.

In [3]:
# The mode flags for the `open` function are "r": read, "t": text mode
stopwords = set([word.lower().strip() for word in open("data/nltk_stopwords.txt", "rt").readlines()])

For extracting words this time, we need to keep hashtags and mentions; and remove stop words.

In [4]:
def extract_words(text, stopwords):
    temp = text.split() # Split the text on whitespace
    text_words = []

    punctuation = set(string.punctuation)
    
    #Keep #tags and @mentions
    punctuation.remove("#")
    punctuation.remove("@")
    
    for word in temp:
        # Remove any punctuation characters present in the beginning of the word
        while len(word) > 0 and word[0] in punctuation:
            word = word[1:]

        # Remove any punctuation characters present in the end of the word
        while len(word) > 0 and word[-1] in punctuation:
            word = word[:-1]

        # Simple rule to eliminate (most) URLs
        if len(word) > 0 and "/" not in word:
            # If it's not a stopword
            if word.lower() not in stopwords:
                # Append this word into our list of words.
                text_words.append(word.lower())

    return text_words


Process tweet data from the CSV file that contains references to Apple. This will be the corpus we will perform analysis on.

In [5]:
tweets = []
line_count = 0

for line in open("data/Apple-Twitter-Sentiment-DFE.csv", "rt"):
    fields = line.strip().split(',')
    
    line_count += 1
    
    # Skip the first line of the file which contains the header
    if line_count == 1:
        continue
    
    text = ",".join(fields[11:])
    
    if len(text) == 0:
        continue
    
    words = extract_words(text, stopwords)
    
    if len(words) > 0:
        tweets.append(words)

    # Limit the corpus
    if len(tweets) == 200:
        break

Define the function to calculate the Inverse Document Frequency for each word and the TFIDF matrix.

In [6]:
def inv_doc_freq(corpus_words):
    number_docs = len(corpus_words)
    
    document_count = {}

    for document in corpus_words:
        word_set = set(document)

        for word in word_set:
            document_count[word] = document_count.get(word, 0) + 1
    
    IDF = {}
    
    for word in document_count:
        IDF[word] = np.log(number_docs/document_count[word])
        
    
    return IDF

def tf_idf(corpus_words):
    IDF = inv_doc_freq(corpus_words)
    
    TFIDF = []

    # Counter is a subclass of `dict` for counting hashable objects
    for document in corpus_words:
        TFIDF.append(Counter(document))
    
    for document in TFIDF:
        for word in document:
            document[word] = document[word]*IDF[word]
            
    return TFIDF

Note that while we call it a matrix, this is effectively a list of dictionaries, which we can consider to be a sparse representation of a matrix.

In [7]:
TFIDF = tf_idf(tweets)

In [8]:
def build_vocabulary(TFIDF):
    words = set()
    
    for document in TFIDF:
        # this operation is union of words and document.keys() and self-assign
        # document.keys() is a set (set operators can only work with set types)
        words |= document.keys()
    
    word_list = list(words)

    # The zip function will generate a tuple for each word that is matched with the index generated by the range function
    word_dict = dict(zip(word_list, range(len(word_list))))
    
    return word_dict, word_list

In [9]:
word_dict, word_list = build_vocabulary(TFIDF)

In [10]:
vocabulary_size = len(word_dict)
print(f"We have {vocabulary_size} words in our vocabulary")

We have 927 words in our vocabulary


Now use the TFIDF matrix and our vocabulary to generate the Term Document matrix. This is just a matter of rearranging the values in our (sparse) TFIDF matrix into the full term document matrix.

In [ ]:
def term_document_matrix(TFIDF, word_list, word_dict):
    vocabulary_size = len(word_dict)
    number_documents = len(TFIDF)
    
    matrix = np.zeros((vocabulary_size, number_documents))
    
    for doc_index in range(number_documents):
        document = TFIDF[doc_index]
        
        for word in document.keys():
            position = word_dict[word]
            
            matrix[position, doc_index] = document[word]
            
    return matrix